# ViT reverse-engineering lab: Colab T4 follow-up studies

This notebook runs the work that did not fit on the 8 GB CPU host:

| Step | Study | Rough T4 time |
|---|---|---|
| 5 | CUDA layer-by-layer equivalence of the educational ViT-Base against the pinned timm reference | 2 min |
| 6 | CUDA inference benchmark: CUDA-event latency and peak allocated memory, 224/384/512 px | 3 min |
| 8 | CIFAR-10 upsampled to 128 px, patch size 8/16/32, width 96, seeds 7/11/19 (9 runs) | 45–70 min |
| 9 | Width-192 patch matrix at native 32 px, patch size 4/8/16, seeds 7/11/19 (9 runs) | 40–60 min |
| 11 | Full-backbone RetinaMNIST fine-tuning (all 86 M parameters, augmentation, fp16 autocast), seeds 7/11/19 | 20–40 min |

**How to run**

1. *Runtime → Change runtime type → T4 GPU*.
2. *Runtime → Run all*. Step 3 asks for Google Drive access, so that finished runs survive a disconnect.
3. If Colab disconnects, reconnect and *Run all* again. Finished runs are skipped, and an interrupted CIFAR run resumes from its last completed epoch.
4. The last cell downloads `colab_t4_results.zip`. Unzip it into `results/colab_t4/` of the repository.

All protocols are frozen in the repository: configs under `configs/experiments/upsampled128/`, `configs/experiments/width192_patch/`, and `configs/experiments/retina_full.yaml`. The CIFAR-10 official test split is never used. RetinaMNIST test predictions are made once per validation-selected model.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/asifuddin01/ViT_reverse.git"
REPO_REF = "main"
WORK_DIR = "/content/ViT_reverse"

USE_DRIVE = True                      # keep runs on Google Drive so a disconnect can resume
DRIVE_DIR = "/content/drive/MyDrive/ViT_reverse_colab"

SEEDS = [7, 11, 19]
RUN_CUDA_EQUIVALENCE = True
RUN_CUDA_BENCHMARK = True
RUN_UPSAMPLED_PATCH_STUDY = True      # 128 px, P = 8/16/32, width 96
RUN_WIDTH192_PATCH_STUDY = True       # 32 px,  P = 4/8/16,  width 192
RUN_RETINA_FULL_FINETUNE = True

SMOKE = False                         # True = a few-minute code-path check, not results

## 2. Clone the repository, install packages, select the GPU

In [ ]:
import os, sys, subprocess, shlex, json, shutil, platform, time
from pathlib import Path


def sh(cmd, cwd=None, check=True):
    """Run a shell command and stream its output into the notebook."""
    print("$", cmd, flush=True)
    process = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if check and code:
        raise RuntimeError(f"Command failed with exit code {code}: {cmd}")
    return code


if not Path(WORK_DIR, ".git").exists():
    sh(f"git clone --branch {shlex.quote(REPO_REF)} {shlex.quote(REPO_URL)} {shlex.quote(WORK_DIR)}")
else:
    sh(f"git -C {shlex.quote(WORK_DIR)} pull --ff-only", check=False)
os.chdir(WORK_DIR)
sh(f"{shlex.quote(sys.executable)} -m pip install -q timm==1.0.30 medmnist==3.0.2")

SRC = str(Path(WORK_DIR, "src"))
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic cuBLAS for reload checks

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda" and not SMOKE:
    raise RuntimeError("No GPU found. Use Runtime > Change runtime type > T4 GPU, then run all again.")
print("Python", platform.python_version(), "| torch", torch.__version__, "| device", DEVICE,
      torch.cuda.get_device_name() if DEVICE == "cuda" else "")

## 3. Result folders (on Google Drive when available)

In [ ]:
TAG = "colab_t4_smoke" if SMOKE else "colab_t4"
OUT = Path("results") / TAG            # small artifacts to bring back into the repository
RUNS = Path("results/runs") / TAG      # full run directories, including checkpoints

if USE_DRIVE and not SMOKE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        print("Not running in Colab; results stay local.")
        USE_DRIVE = False

if USE_DRIVE and not SMOKE:
    for local, remote in ((OUT, Path(DRIVE_DIR, "artifacts")), (RUNS, Path(DRIVE_DIR, "runs"))):
        remote.mkdir(parents=True, exist_ok=True)
        local.parent.mkdir(parents=True, exist_ok=True)
        if local.is_symlink():
            continue
        if local.exists():
            shutil.copytree(local, remote, dirs_exist_ok=True)
            shutil.rmtree(local)
        local.symlink_to(remote.resolve(), target_is_directory=True)
        print(f"{local} -> {remote}")

OUT.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
print("artifacts:", OUT.resolve())
print("runs:", RUNS.resolve())

## 4. Datasets and environment record

In [ ]:
import hashlib
from importlib.metadata import version
from torchvision.datasets import CIFAR10

cifar = CIFAR10("data", train=True, download=True)
cifar[0][0].save("data/cifar_train_00000.png")   # the real image used by the equivalence check

if RUN_RETINA_FULL_FINETUNE:
    from medmnist import RetinaMNIST
    for split in ("train", "val", "test"):
        RetinaMNIST(split=split, size=224, root="data", download=True)
    retina_md5 = hashlib.md5(Path("data/retinamnist_224.npz").read_bytes()).hexdigest()
    assert retina_md5 == "eae7e3b6f3fcbda4ae613ebdcbe35348", f"Unexpected RetinaMNIST file: {retina_md5}"

nvidia_smi = (subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
              if shutil.which("nvidia-smi") else None)
ENVIRONMENT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torchvision": version("torchvision"),
    "timm": version("timm"),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name() if DEVICE == "cuda" else None,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version() if DEVICE == "cuda" else None,
    "git_commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                                 text=True).stdout.strip(),
    "smoke": SMOKE,
    "nvidia_smi": nvidia_smi,
}
(OUT / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2) + "\n")
print(json.dumps({k: v for k, v in ENVIRONMENT.items() if k != "nvidia_smi"}, indent=2))

## 5. Equivalence on the GPU

The same 4 inputs × 25 stages as the CPU report (`results/tables/equivalence.csv`). The reference is built with fused attention disabled. Each stage is checked against `rtol=1e-4, atol=1e-5`, and its actual errors are saved even if the tolerance is missed: GPU kernels are not bit-identical to CPU ones.

In [ ]:
if RUN_CUDA_EQUIVALENCE:
    import yaml
    from vit_lab.reference.compare import compare_reference

    base_cfg = yaml.safe_load(Path("configs/vit_base.yaml").read_text())
    rows = compare_reference(base_cfg, output=OUT / "equivalence_cuda.csv",
                             mapping_output=OUT / "weight_mapping_cuda.csv",
                             image_path=Path("data/cifar_train_00000.png"), device=DEVICE)
    worst = max(rows, key=lambda row: row["max_absolute_error"])
    print(f"{sum(row['passed'] for row in rows)} / {len(rows)} stage comparisons within tolerance")
    print(f"largest absolute difference: {worst['max_absolute_error']:.3e} "
          f"({worst['sample']}, {worst['stage']})")
    for row in rows:
        if row["stage"] == "head":
            print(f"  logits {row['sample']:<28} max abs {row['max_absolute_error']:.3e} "
                  f"cosine {row['cosine_similarity']:.8f} passed={row['passed']}")
    del rows
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 6. GPU inference benchmark

FP32, `eval()` and `inference_mode()`, 20 warm-ups then 100 timed forwards per case. Each forward is timed with CUDA events and synchronized. Memory is `torch.cuda.max_memory_allocated` / `max_memory_reserved` over warm-up and trials, including the weights. MACs are analytical, with the same convention as `docs/benchmarking.md`.

In [ ]:
if RUN_CUDA_BENCHMARK:
    import csv, gc, statistics
    import yaml
    from vit_lab.benchmarking.runner import _load_model, analytical_macs

    base_cfg = yaml.safe_load(Path("configs/vit_base.yaml").read_text())
    warmups, trials = (1, 2) if SMOKE else (20, 100)
    cases = [("educational", 224, 1), ("educational", 224, 8), ("educational", 384, 1),
             ("educational", 512, 1), ("timm_reference", 224, 1), ("timm_reference", 224, 8)]
    bench_rows = []
    for implementation, resolution, batch in cases:
        model, checksum = _load_model(base_cfg, implementation, False)
        model = model.to(DEVICE).eval()
        kernel = (("timm_fused" if model.blocks[0].attn.fused_attn else "timm_manual")
                  if implementation == "timm_reference" else "educational_manual")
        inputs = torch.randn(batch, 3, resolution, resolution, device=DEVICE)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        latencies = []
        with torch.inference_mode():
            for _ in range(warmups):
                model(inputs)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            for _ in range(trials):
                if DEVICE == "cuda":
                    start = torch.cuda.Event(enable_timing=True)
                    end = torch.cuda.Event(enable_timing=True)
                    start.record()
                    model(inputs)
                    end.record()
                    torch.cuda.synchronize()
                    latencies.append(start.elapsed_time(end))
                else:
                    begin = time.perf_counter()
                    model(inputs)
                    latencies.append((time.perf_counter() - begin) * 1000)
        counts = analytical_macs(base_cfg["model"], resolution)
        median = statistics.median(latencies)
        row = {
            "implementation": implementation, "attention_kernel": kernel,
            "device": DEVICE, "gpu": ENVIRONMENT["gpu"], "torch": torch.__version__,
            "dtype": "float32", "batch_size": batch, "resolution": resolution, "patch_size": 16,
            "tokens": counts["tokens"], "parameter_count": sum(p.numel() for p in model.parameters()),
            "macs_per_image": counts["macs_per_image"], "flops_2_per_mac": counts["flops_2_per_mac"],
            "timing_method": "CUDA events per forward, synchronized" if DEVICE == "cuda" else "perf_counter",
            "warmups": warmups, "trials": trials,
            "median_latency_ms": median, "minimum_latency_ms": min(latencies),
            "p90_latency_ms": sorted(latencies)[int(0.9 * (len(latencies) - 1))],
            "std_latency_ms": statistics.stdev(latencies),
            "throughput_images_per_second": batch * 1000 / median,
            "peak_allocated_bytes": torch.cuda.max_memory_allocated() if DEVICE == "cuda" else None,
            "peak_reserved_bytes": torch.cuda.max_memory_reserved() if DEVICE == "cuda" else None,
            "checkpoint_sha256": checksum,
        }
        bench_rows.append({**row, "trial_latencies_ms": latencies})
        peak = row["peak_allocated_bytes"]
        print(f"{implementation:<15} {resolution} px batch {batch}: {median:8.2f} ms median, "
              f"{row['throughput_images_per_second']:7.1f} img/s"
              + (f", peak allocated {peak / 2**20:7.0f} MiB" if peak else ""))
        del model, inputs
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    with (OUT / "benchmark_cuda.csv").open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=[k for k in bench_rows[0] if k != "trial_latencies_ms"],
                                lineterminator="\n")
        writer.writeheader()
        writer.writerows({k: v for k, v in r.items() if k != "trial_latencies_ms"} for r in bench_rows)
    (OUT / "benchmark_cuda.json").write_text(json.dumps(
        {"environment": ENVIRONMENT, "cases": bench_rows}, indent=2) + "\n")

## 7. CIFAR-10 training helpers

Every case is trained with the repository CLI (`python -m vit_lab.cli train`). The CPU studies used the same trainer, split rule, augmentation, AdamW settings, 20-epoch schedule, and best-validation checkpoint. For each case the helper:

- checks the frozen config,
- resumes an interrupted run,
- requires the reloaded best checkpoint to reproduce the saved validation predictions,
- checks that the variants of one seed used the same train/validation split.

In [ ]:
import copy, csv, hashlib, statistics
import yaml
from vit_lab.benchmarking.runner import analytical_macs
from vit_lab.models.vision_transformer import VisionTransformer

BASE = yaml.safe_load(Path("configs/experiments/patch_size/patch_p08_seed7.yaml").read_text())
STUDY_SEEDS = SEEDS[:2] if SMOKE else SEEDS

# Unfinished runs from commit 3cea24b upsampled each image on the CPU (superseded, and it crashed
# at the final reload check). Restart them under the GPU per-batch upsampling protocol.
SUPERSEDED_COMMITS = {"3cea24bbcebd8f5c23dafb0c8d0ede3ab3ec51de"}
for environment_file in (RUNS / "upsampled128").glob("*/environment.json"):
    run_dir = environment_file.parent
    if (not (run_dir / "summary.json").exists()
            and json.loads(environment_file.read_text()).get("git_commit") in SUPERSEDED_COMMITS):
        print("discarding superseded unfinished run:", run_dir)
        shutil.rmtree(run_dir)


def frozen_config(study, label, overrides, seed):
    config = copy.deepcopy(BASE)
    config["model"].update(overrides)
    config["training"]["seed"] = seed
    path = Path("configs/experiments") / study / f"{label}_seed{seed}.yaml"
    if path.exists():
        if yaml.safe_load(path.read_text()) != config:
            raise ValueError(f"Frozen config differs from the expected variant: {path}")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(yaml.safe_dump(config, sort_keys=False))
    return path, config


def train_case(study, label, overrides, seed):
    config_path, config = frozen_config(study, label, overrides, seed)
    run_dir = RUNS / study / f"{label}_seed{seed}"
    if not (run_dir / "summary.json").exists():
        if run_dir.exists() and not (run_dir / "last.pt").exists():
            shutil.rmtree(run_dir)   # stopped before the first epoch finished
        command = [sys.executable, "-m", "vit_lab.cli", "train", "--config", str(config_path),
                   "--run-dir", str(run_dir), "--device", DEVICE]
        if run_dir.exists():
            command.append("--resume")
        if SMOKE:
            command += ["--epochs", "1", "--max-train-batches", "2", "--max-val-batches", "2"]
        sh(shlex.join(command))
    summary = json.loads((run_dir / "summary.json").read_text())
    environment = json.loads((run_dir / "environment.json").read_text())
    with (run_dir / "metrics.csv").open(newline="") as handle:
        history = list(csv.DictReader(handle))
    expected_epochs = 1 if SMOKE else config["training"]["epochs"]
    if len(history) != expected_epochs or int(history[-1]["epoch"]) != expected_epochs:
        raise ValueError(f"Incomplete epoch history: {run_dir}")
    if not summary["validation_predictions_identical_after_reload"]:
        raise ValueError(f"Reloaded checkpoint changed validation predictions: {run_dir}")
    split = json.loads((run_dir / "split_indices.json").read_text())
    target = OUT / study
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy(run_dir / "metrics.csv", target / f"{label}_seed{seed}_metrics.csv")
    return {
        "study": study, "label": label, "seed": seed,
        "image_size": config["model"]["image_size"], "patch_size": config["model"]["patch_size"],
        "embed_dim": config["model"]["embed_dim"],
        "best_validation_accuracy": summary["best_validation_accuracy"],
        "best_epoch": summary["best_epoch"],
        "reload_max_abs_logit_difference": summary.get("reload_max_abs_logit_difference"),
        "total_epoch_seconds": sum(float(row["seconds"]) for row in history),
        "split_sha256": hashlib.sha256(json.dumps(split, sort_keys=True).encode()).hexdigest(),
        "config_path": str(config_path),
        "config_sha256": hashlib.sha256(config_path.read_bytes()).hexdigest(),
        "device": environment["device"], "gpu": environment.get("gpu"),
        "git_commit": environment["git_commit"], "smoke_run": summary["smoke_run"],
    }


def write_rows(path, rows):
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


def run_study(study, variants):
    """Seed-major order, so each finished seed is a complete paired comparison."""
    rows = []
    for seed in STUDY_SEEDS:
        for label, overrides in variants:
            rows.append(train_case(study, label, overrides, seed))
            write_rows(OUT / study / "runs.csv", rows)
        if len({row["split_sha256"] for row in rows if row["seed"] == seed}) != 1:
            raise ValueError(f"Seed {seed} variants did not share one train/validation split")
    aggregate = []
    for label, overrides in variants:
        selected = [row for row in rows if row["label"] == label]
        values = [row["best_validation_accuracy"] * 100 for row in selected]
        model_cfg = {**BASE["model"], **overrides}
        counts = analytical_macs(model_cfg, model_cfg["image_size"])
        aggregate.append({
            "label": label, **overrides, "tokens": counts["tokens"],
            "parameters": sum(p.numel() for p in VisionTransformer(**model_cfg).parameters()),
            "macs_per_image": counts["macs_per_image"],
            "seeds": ",".join(str(row["seed"]) for row in selected),
            "validation_accuracy_mean_pct": statistics.mean(values),
            "validation_accuracy_sample_sd_pct": statistics.stdev(values) if len(values) > 1 else None,
            "per_seed_pct": " / ".join(f"{v:.2f}" for v in values),
            "total_epoch_minutes": sum(row["total_epoch_seconds"] for row in selected) / 60,
        })
    write_rows(OUT / study / "aggregate.csv", aggregate)
    for row in aggregate:
        sd = row["validation_accuracy_sample_sd_pct"]
        print(f"{row['label']:<11} tokens {row['tokens']:>4}  params {row['parameters']:>9,}  "
              f"MACs {row['macs_per_image'] / 1e6:7.2f} M  val acc {row['validation_accuracy_mean_pct']:.2f}"
              + (f" ± {sd:.2f}" if sd is not None else "") + f"  [{row['per_seed_pct']}]")
    return rows, aggregate


def plot_study(study, aggregate, rows, title, note):
    import matplotlib.pyplot as plt

    labels = [row["label"] for row in aggregate]
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    for x, label in enumerate(labels):
        points = [r["best_validation_accuracy"] * 100 for r in rows if r["label"] == label]
        ax.scatter([x] * len(points), points, color="#888888", zorder=3, label="seed" if x == 0 else None)
    means = [row["validation_accuracy_mean_pct"] for row in aggregate]
    sds = [row["validation_accuracy_sample_sd_pct"] or 0 for row in aggregate]
    ax.errorbar(range(len(labels)), means, yerr=sds, fmt="o-", capsize=6, color="#1767b3",
                ecolor="#dd6b20", linewidth=2, label="mean ± sample SD")
    ax.set_xticks(range(len(labels)),
                  [f"{row['label']}\n{row['tokens']} tokens" for row in aggregate])
    ax.set_ylabel("Best validation accuracy (%)")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="best", fontsize=8)
    fig.text(0.5, 0.01, note, ha="center", fontsize=8)
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    for suffix in ("png", "svg"):
        fig.savefig(OUT / study / f"{study}_accuracy.{suffix}", dpi=160)
    plt.show()

## 8. CIFAR-10 upsampled to 128 px: patch size 8, 16, 32

The CIFAR-10 examples, split rule, and model (width 96, depth 4, 8 heads) are the same as the native study. Only `image_size=128` changes, together with the patch size. Each batch is upsampled bicubically on the GPU, after the usual 32-px crop and flip. That gives 256, 64, and 16 patches. **Upsampling adds tokens and compute, not image detail.** Accuracies here are exploratory and are mainly a cost-versus-token study. `img128_p32` has the same 16-patch grid as native `P=8`, which allows a direct comparison with the CPU result of 63.40 ± 1.15%.

In [ ]:
if RUN_UPSAMPLED_PATCH_STUDY:
    upsampled_variants = [(f"img128_p{p:02d}", {"image_size": 128, "patch_size": p}) for p in (8, 16, 32)]
    upsampled_rows, upsampled_aggregate = run_study("upsampled128", upsampled_variants)
    plot_study("upsampled128", upsampled_aggregate, upsampled_rows,
               "CIFAR-10 upsampled to 128 px: patch size",
               "20 epochs · width 96 · depth 4 · 8 heads · validation only · upsampling adds no detail")

## 9. Width 192 at native 32 px: patch size 4, 8, 16

This repeats the width-96 patch-size study at width 192. `w192_p08` uses exactly the same config as the CPU width-192 runs, so the cell also shows the GPU and CPU results for the same seed side by side as a cross-device check. GPU floating-point results are not bit-identical to CPU, so small differences are expected.

In [ ]:
if RUN_WIDTH192_PATCH_STUDY:
    width_variants = [(f"w192_p{p:02d}", {"embed_dim": 192, "patch_size": p}) for p in (4, 8, 16)]
    width_rows, width_aggregate = run_study("width192_patch", width_variants)
    plot_study("width192_patch", width_aggregate, width_rows,
               "Width 192 at native 32 px: patch size",
               "20 epochs · depth 4 · 8 heads · validation only")
    cpu_dir = Path("results/ablations/width")
    for row in width_rows:
        cpu_file = cpu_dir / f"width_d192_seed{row['seed']}_summary.json"
        if row["label"] == "w192_p08" and cpu_file.exists():
            cpu = json.loads(cpu_file.read_text())
            print(f"seed {row['seed']}: {row['device']} {row['best_validation_accuracy'] * 100:.2f}% "
                  f"vs CPU {cpu['best_validation_accuracy'] * 100:.2f}%; same split: "
                  f"{row['split_sha256'] == cpu['split_sha256']}")

## 10. Full-backbone RetinaMNIST fine-tuning

This follows the frozen protocol in `configs/experiments/retina_full.yaml`:

- Starts from the verified ViT-Base, with a new 5-class head.
- All parameters train: AdamW at learning rate 2e-5, weight decay 0.05, batch 16, 10 epochs, one warm-up epoch then cosine decay.
- Augmentation: random resized crop (scale 0.8–1.0) and horizontal flip. Training uses fp16 autocast.
- The epoch with the best validation QWK is kept. The test set is predicted once for that model.
- All metrics are recomputed from the saved predictions, with 2,000-resample bootstrap 95% intervals.

The CPU partial fine-tune and baselines are printed alongside for comparison.

In [ ]:
if RUN_RETINA_FULL_FINETUNE:
    import csv
    from vit_lab.experiments import retina, retina_full

    retina_dir = OUT / "retina_full"
    retina_rows = retina_full.run(Path("configs/experiments/retina_full.yaml"), device=DEVICE,
                                  offline=False, result_dir=retina_dir, smoke=SMOKE)
    retina.plot(retina_dir, retina_dir / "retina_full_transfer.png")
    with Path("results/transfer/retina/summary.csv").open(newline="") as handle:
        cpu_rows = list(csv.DictReader(handle))
    print(f"{'method (test)':<28}{'acc':>8}{'bal acc':>9}{'macro F1':>10}{'QWK':>8}   QWK 95% CI")
    for row in cpu_rows + [{k: str(v) for k, v in r.items()} for r in retina_rows]:
        name = row["method"] if row["method"].startswith("full") else "cpu " + row["method"]
        print(f"{name:<28}{float(row['test_accuracy']):8.3f}{float(row['test_balanced_accuracy']):9.3f}"
              f"{float(row['test_macro_f1']):10.3f}{float(row['test_quadratic_weighted_kappa']):8.3f}"
              f"   [{float(row['test_quadratic_weighted_kappa_ci_low']):.3f}, "
              f"{float(row['test_quadratic_weighted_kappa_ci_high']):.3f}]")
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 11. Download the results

In [ ]:
bundle = OUT / "configs"
for study in ("upsampled128", "width192_patch"):
    source = Path("configs/experiments") / study
    if source.exists():
        shutil.copytree(source, bundle / study, dirs_exist_ok=True)
shutil.copy("configs/experiments/retina_full.yaml", bundle / "retina_full.yaml")

archive = shutil.make_archive(str(Path(WORK_DIR).parent / f"{TAG}_results"), "zip", root_dir=OUT)
print("bundle:", archive)
for path in sorted(OUT.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUT)}  ({path.stat().st_size / 1024:.1f} KiB)")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

## Bringing the results back

1. Unzip `colab_t4_results.zip` into `results/colab_t4/` of the repository, so that `results/colab_t4/environment.json` exists. Leave out `configs/`: the same configs are already committed.
2. Commit and push.
3. Ask for the results to be written up in `docs/experiments.md`, `docs/benchmarking.md`, `README.md`, and `PROJECT_STATUS.md`.

Full run directories with checkpoints stay on Google Drive under `ViT_reverse_colab/runs`. They are too large for Git.